# OpportunityLens — Interactive Notebook Demo

This notebook demonstrates the **offline, local-catalog mode** of OpportunityLens. It reads the bundled sample JSON, validates the catalog, and searches it deterministically. It does **not** call Amazon S3 or Amazon Bedrock unless you explicitly run the optional live S3 cell with AWS access configured.

Repository: https://github.com/leelasreeamrutha99-glitch/AWS-PROJECT

## Run it
From the repository root, install the project dependencies with `python -m pip install -r requirements.txt`, then open this notebook in Jupyter. If you open the notebook with the working directory set to `notebooks/`, the path setup below also handles that.

In [ ]:
from pathlib import Path
import json
import os
import sys
from pprint import pprint

ROOT = Path.cwd()
if not (ROOT / "data" / "learning_resources.json").exists():
    ROOT = ROOT.parent
if not (ROOT / "data" / "learning_resources.json").exists():
    raise FileNotFoundError("Open this notebook from the repository root or its notebooks/ folder.")
sys.path.insert(0, str(ROOT))

from catalog_tool import validate_catalog, search_catalog

catalog_path = ROOT / "data" / "learning_resources.json"
catalog = json.loads(catalog_path.read_text(encoding="utf-8"))
validated = validate_catalog(catalog)
print(f"Catalog: {validated['catalog_id']}")
print(f"Validated resources: {len(validated['resources'])}")
print(f"Catalog updated on: {validated['catalog_updated_on']}")

## 1. Search for free data-science resources

In [ ]:
result = search_catalog(
    validated,
    skill_query="data science",
    budget_preference="free",
    format_preference="any",
    max_results=5,
)
print(result["message"])
print(f"Matches: {result['result_count']}")
for resource in result["results"]:
    print(f"\n{resource['title']} ({resource['id']})")
    print(f"Provider: {resource['provider']} | Level: {resource['level']} | Format: {resource['format']}")
    print(f"Cost label: {resource['cost_label']}")
    print(f"Review status: {resource['review_status']} (review date: {resource['reviewed_on']})")
    print(f"Official URL: {resource['official_url']}")

## 2. Search by skill and format

In [ ]:
python_docs = search_catalog(
    validated,
    skill_query="Python",
    budget_preference="any",
    format_preference="documentation",
    max_results=5,
)
pprint(python_docs)

## 3. Deliberate no-match test

An unknown topic should yield no matching entries rather than a made-up course or link.

In [ ]:
unknown = search_catalog(
    validated,
    skill_query="quantum underwater basket weaving",
    budget_preference="any",
    format_preference="any",
    max_results=5,
)
print(unknown["message"])
print(f"Matches: {unknown['result_count']}")
assert unknown["results"] == [], "Unknown query unexpectedly returned catalog entries"
print("PASS: no fabricated fallback was returned.")

## 4. Optional: read the configured private S3 catalog

This cell is skipped unless `LEARNING_CATALOG_BUCKET` is set and the runtime already has AWS credentials/role access. Prefer an IAM workload role; never paste AWS access keys into a notebook or commit them to GitHub. A successful local notebook run is not evidence of live Bedrock integration.

In [ ]:
if os.environ.get("LEARNING_CATALOG_BUCKET"):
    from catalog_tool import load_catalog_from_s3
    live_catalog = load_catalog_from_s3()
    print("S3 object read succeeded.")
    pprint(live_catalog["source"])
    print(f"Validated live resources: {len(live_catalog['catalog']['resources'])}")
else:
    print("SKIPPED: LEARNING_CATALOG_BUCKET is not configured. No AWS call was made.")

## What this notebook proves — and what it does not

- It demonstrates catalog validation and deterministic search against the **bundled sample data**.
- It checks that an unmatched query returns no fallback resource.
- The optional S3 cell attempts a live read only if configured.
- It does not prove a live Amazon Bedrock agent/tool invocation, hosted app availability, current URL uptime, prices, or course quality.
- For the challenge video, record the actual behavior and clearly label local mode versus live AWS mode.